# Task 3: Build Rule-Based Linguistic Features

Adds fast, free, non-API-dependent features to `trends_combined_english.csv`, so they're
ready to use in Task 5's regression regardless of how Task 4 (zero-shot) goes.

In [1]:
import pandas as pd
import numpy as np
import ast
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import emoji

## Step 1: Load `trends_combined_english.csv`

In [2]:
combined = pd.read_csv("../../output/cleaned_data/trends_combined_english.csv")
print(combined.shape)

(126410, 23)


## Step 2: Word count and log-transformed word count

In [3]:
combined["word_count"] = combined["text"].str.split().str.len()
combined["log_word_count"] = np.log1p(combined["word_count"])

print("Raw word_count skewness:", combined["word_count"].skew())
print("Log word_count skewness:", combined["log_word_count"].skew())

Raw word_count skewness: 1.6923324839998368
Log word_count skewness: -0.39836319860553915


Confirming against expected: raw skewness ~1.692 (substantially right-skewed), log-transformed
skewness ~-0.398 (close to symmetric). `log_word_count` is the actual regression predictor.

## Step 3: Sentiment score via VADER

In [4]:
analyzer = SentimentIntensityAnalyzer()
combined["sentiment_score"] = combined["text"].apply(lambda t: analyzer.polarity_scores(t)["compound"])

print(combined["sentiment_score"].describe())

count    126410.000000
mean          0.620119
std           0.420554
min          -0.998100
25%           0.381800
50%           0.817200
75%           0.951000
max           0.999800
Name: sentiment_score, dtype: float64


## Step 4: Hashtag count, treating null as 0

Checked real `hashtags` values first: this project's `hashtags` column is a stringified Python
list (e.g. `["quarantinebaking","baking",...]`), not a plain comma-separated string. All 112,922
non-null values in `trends_combined_english.csv` were confirmed to parse cleanly via
`ast.literal_eval` (0 parse failures), so that's used instead of a plain `.split(",")`.

In [5]:
def count_hashtags(h):
    if pd.isna(h):
        return 0
    parsed = ast.literal_eval(h)
    return len(parsed)

combined["hashtag_count"] = combined["hashtags"].apply(count_hashtags)

print(combined["hashtag_count"].describe())
print("Rows with 0 hashtags (includes null-hashtag rows):", (combined["hashtag_count"] == 0).sum())

count    126410.000000
mean         16.200791
std          10.762886
min           0.000000
25%           6.000000
50%          17.000000
75%          27.000000
max          51.000000
Name: hashtag_count, dtype: float64
Rows with 0 hashtags (includes null-hashtag rows): 13488


## Step 5: Punctuation and emoji features

In [6]:
combined["exclamation_count"] = combined["text"].str.count("!")
combined["question_count"] = combined["text"].str.count(r"\?")
combined["emoji_count"] = combined["text"].apply(lambda t: emoji.emoji_count(str(t)))

print(combined[["exclamation_count", "question_count", "emoji_count"]].describe())
print()
pct_with_emoji = (combined["emoji_count"] > 0).mean() * 100
print(f"Percent of posts with at least 1 emoji: {pct_with_emoji:.1f}%")
print(f"Mean emoji count (posts with >=1 emoji): {combined.loc[combined['emoji_count'] > 0, 'emoji_count'].mean():.2f}")

       exclamation_count  question_count    emoji_count
count      126410.000000   126410.000000  126410.000000
mean            1.226817        0.307491       2.430947
std             1.955575        0.766426       4.125851
min             0.000000        0.000000       0.000000
25%             0.000000        0.000000       0.000000
50%             0.000000        0.000000       1.000000
75%             2.000000        0.000000       3.000000
max            29.000000       22.000000     148.000000

Percent of posts with at least 1 emoji: 62.3%
Mean emoji count (posts with >=1 emoji): 3.90


## Step 6: Log-transformed outcome variables

In [7]:
combined["log_likes"] = np.log1p(combined["statistics.like_count"])
combined["log_comments"] = np.log1p(combined["statistics.comment_count"])

## Step 7: Confirm `is_covid_framed` already exists, no changes needed

In [8]:
print("is_covid_framed" in combined.columns)
print(combined["is_covid_framed"].dtype)
print(combined["is_covid_framed"].value_counts())

True
bool
is_covid_framed
False    112153
True      14257
Name: count, dtype: int64


## Step 8: Save the updated file

In [9]:
combined.to_csv("../../output/cleaned_data/trends_combined_english_features.csv", index=False)
print(f"Saved trends_combined_english_features.csv, shape: {combined.shape}")
print(combined.columns.tolist())

Saved trends_combined_english_features.csv, shape: (126410, 32)
['content_type', 'creation_time', 'hashtags', 'id', 'is_branded_content', 'lang', 'match_type', 'mcl_url', 'modified_time', 'multimedia', 'post_owner.id', 'post_owner.name', 'post_owner.type', 'post_owner.username', 'statistics.comment_count', 'statistics.like_count', 'statistics.views', 'statistics.views_date_last_refreshed', 'text', 'date_parsed', 'month', 'is_covid_framed', 'trend', 'word_count', 'log_word_count', 'sentiment_score', 'hashtag_count', 'exclamation_count', 'question_count', 'emoji_count', 'log_likes', 'log_comments']
